# Step 1 — Data Ingestion & Search Index Setup

This step prepares the drugstore dataset for retrieval. It loads the Excel file into a Pandas DataFrame, cleans the column names, and extracts prices into a numeric column so they can be used for calculations and filtering. Each product is also converted into a text document. Two complementary search indexes are then created: FAISS for semantic similarity and BM25 for keyword-based matching. These indexes provide the retrieval foundation for the following RAG and agent steps.

In [ ]:
# ============================================================
# Step 1: Ingest Data & Setup Search Indices
# ============================================================

!pip -q install pandas openpyxl sentence-transformers faiss-cpu rank-bm25

import os, re, numpy as np, pandas as pd, faiss
from google.colab import drive
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

drive.mount("/content/drive")
FILE_PATH = "/content/drive/MyDrive/drugstore_100_items.xlsx"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

df = pd.read_excel(FILE_PATH)
df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]

# Parse numeric price for programmatic table queries
for col in df.columns:
    if any(k in col.lower() for k in ["price", "cost", "rate", "ils"]):
        df["_numeric_price"] = pd.to_numeric(
            df[col].astype(str).str.replace(r"[^\d.]", "", regex=True).str.strip(),
            errors="coerce"
        )
        break

# Convert rows to text records for semantic search
documents = []
for _, row in df.iterrows():
    documents.append("\n".join([f"{c}: {v}" for c, v in row.items() if not str(c).startswith("_") and pd.notna(v)]))

# FAISS Vector Search
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedder.encode(documents, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# BM25 Lexical Search
bm25 = BM25Okapi([doc.lower().split() for doc in documents])

print(f"✅ Ingested {len(df)} products into DataFrame and Vector Store.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.1 MB/s eta 0:00:00
Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Ingested 100 products into DataFrame and Vector Store.


#Step 2 — Load & Quantize the LLM

This step loads the Qwen 2.5 7B Instruct language model and its tokenizer. The model is loaded using 4-bit NF4 quantization, which significantly reduces GPU memory requirements compared with full-precision inference. This makes it possible to run the 7B-parameter model efficiently on a T4 GPU while still using it for reasoning, tool selection, and answer generation.

In [ ]:
# ============================================================
# Step 2: Load 7B Model in 4-bit (Fast on T4 GPU)
# ============================================================

!pip -q install transformers accelerate bitsandbytes
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
    device_map="auto"
)

print(f"✅ Model loaded on {model.device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded on cuda:0


#Step 3 — Tool Routing & QA Pipeline

This step creates the first RAG-based question-answering pipeline. The LLM acts as a router and determines whether a question should be answered using an exact table query or semantic search. Table queries are used for numerical and structured questions such as counts, prices, averages, and filters, while semantic search is used for qualitative questions such as symptoms, product uses, or cleansing. After retrieving the relevant database information, the LLM generates a concise answer based only on the retrieved results.

In [ ]:
# ============================================================
# Step 3: Tool Execution Agent + QA Pipeline
# ============================================================

import json

def semantic_search(query, top_k=15):
    """Retrieves semantic matches for health/symptom/qualitative questions."""
    q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, faiss_idx = index.search(q_vec, len(documents))
    bm25_scores = bm25.get_scores(query.lower().split())

    combined = {idx: 0.6 * (1 / (rank + 1)) + 0.4 * bm25_scores[idx] for rank, idx in enumerate(faiss_idx[0])}
    top_indices = sorted(combined, key=combined.get, reverse=True)[:top_k]
    return "\n\n".join([documents[i] for i in top_indices])


def query_table(python_code):
    """Executes Pandas code directly against the entire DataFrame for exact math/filtering."""
    try:
        local_vars = {"df": df, "pd": pd, "np": np}
        # Safely evaluate or execute Python code
        exec(f"result = {python_code}", {}, local_vars)
        res = local_vars["result"]

        if isinstance(res, (pd.DataFrame, pd.Series)):
            clean_res = res.drop(columns=[c for c in res.columns if c.startswith("_")], errors="ignore") if isinstance(res, pd.DataFrame) else res
            return clean_res.to_string()
        return str(res)
    except Exception as e:
        return f"Error executing table query: {e}"


def ask_rag(question):
    # ------------------------------------------------------------
    # ROUTER STEP: LLM decides whether to query the table or vector search
    # ------------------------------------------------------------
    display_cols = [c for c in df.columns if not c.startswith("_")]

    router_prompt = [
        {
            "role": "system",
            "content": (
                "You are an AI Tool Router for a product database.\n"
                f"DataFrame `df` columns: {display_cols}. Numeric price column: '_numeric_price'.\n\n"
                "You must select ONE tool by outputting a JSON object:\n\n"
                "1. `query_table`: Use for quantitative queries (counts, price bounds like < 20, average price, min/max, exact product price lookups).\n"
                "   Example JSON: {\"tool\": \"query_table\", \"code\": \"df[df['_numeric_price'] < 20][['Item', 'Typical Price (ILS)']]\"}\n"
                "   Example JSON: {\"tool\": \"query_table\", \"code\": \"len(df)\"}\n"
                "   Example JSON: {\"tool\": \"query_table\", \"code\": \"df['_numeric_price'].mean()\"}\n"
                "   Example JSON: {\"tool\": \"query_table\", \"code\": \"df.loc[df['_numeric_price'].idxmax()][['Item', 'Typical Price (ILS)']]\"}\n\n"
                "2. `semantic_search`: Use for qualitative/medical queries (symptoms, headache, allergies, cleansing, heartburn).\n"
                "   Example JSON: {\"tool\": \"semantic_search\"}\n\n"
                "Respond with ONLY a valid JSON object."
            )
        },
        {"role": "user", "content": f"User Question: {question}"}
    ]

    prompt = tokenizer.apply_chat_template(router_prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    router_raw = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Parse decision
    tool_call = {"tool": "semantic_search"}
    try:
        json_match = re.search(r"\{.*\}", router_raw, re.DOTALL)
        if json_match:
            tool_call = json.loads(json_match.group(0))
    except Exception:
        tool_call = {"tool": "semantic_search"}

    # ------------------------------------------------------------
    # EXECUTION PATH A: Table Execution (Math, Filters, Counts)
    # ------------------------------------------------------------
    if tool_call.get("tool") == "query_table" and "code" in tool_call:
        table_output = query_table(tool_call["code"])

        summary_prompt = [
            {"role": "system", "content": "Summarize the database query result clearly and concisely for the user."},
            {"role": "user", "content": f"User Question: {question}\nDatabase Output:\n{table_output}"}
        ]

        prompt = tokenizer.apply_chat_template(summary_prompt, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False)

        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # ------------------------------------------------------------
    # EXECUTION PATH B: Semantic Search (Medical, Symptoms, Uses)
    # ------------------------------------------------------------
    retrieved_context = semantic_search(question, top_k=15)

    qa_prompt = [
        {
            "role": "system",
            "content": (
                "You are an expert drugstore database assistant.\n"
                "Answer the user query concisely using ONLY the provided database records.\n"
                "List matching items clearly with their exact names and prices.\n"
                "Do NOT invent facts or recommend heartburn/digestive products for headaches."
            )
        },
        {"role": "user", "content": f"DATABASE RECORDS:\n{retrieved_context}\n\nUSER QUESTION:\n{question}"}
    ]

    prompt = tokenizer.apply_chat_template(qa_prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# ============================================================
# Test Execution
# ============================================================

questions = [
    "I have a headache, what medication do you have for me?",
    "What products are used for cleansing?",
    "How much does the most expensive product cost?",
    "Which products cost less than 20 ILS?",
    "How many total products are in the database?",
    "What is the price of Nurofen?",
    "What is the average price of an item?"
]

for q in questions:
    print(f"\n❓ Question: {q}")
    print(f"💡 Answer:\n{ask_rag(q)}")
    print("-" * 60)


❓ Question: I have a headache, what medication do you have for me?
💡 Answer:
For headache relief, you can choose from the following medications:

- Nurofen 200 mg Tablets: ₪22
- Advil 200 mg Tablets: ₪18
- Acamol 500 mg Tablets: ₪14
- Tylenol Extra Strength: ₪24
- Aspirin 100 mg: ₪12

These options contain ibuprofen, paracetamol, or aspirin, which are commonly used for pain and fever relief.
------------------------------------------------------------

❓ Question: What products are used for cleansing?
💡 Answer:
Here are the products used for cleansing:

- Dove Deeply Nourishing Body Wash (Body cleanser, 500 ml, ₪18)
- Garnier Micellar Water (Makeup-removing cleanser, 400 ml, ₪25)
- Hydrogen Peroxide 3% (Topical cleansing solution, 250 ml, ₪10)
- La Roche-Posay Toleriane Cleanser (Facial cleanser, 200 ml, ₪70)
- CeraVe Foaming Cleanser (Facial cleanser, 236 ml, ₪55)
- Cetaphil Gentle Skin Cleanser (Facial cleanser, 236 ml, ₪55)
- Sterimar Nasal Spray (Nasal cleansing, 50 ml, ₪32)
-----

#Example: A Limitation of the RAG Pipeline

The following example demonstrates a query that the RAG pipeline does not handle correctly. The question requires both semantic reasoning (identifying products suitable for a headache) and quantitative reasoning (finding the cheapest matching product).

To handle queries that require multiple types of operations, we introduced a multi-tool agent that can select and combine the appropriate tools.

In [ ]:
q = "what is the cheapest thing you have for an headache?"
print(f"\n❓ Question: {q}")
print(f"💡 Answer:\n{ask_rag(q)}")


❓ Question: what is the cheapest thing you have for an headache?
💡 Answer:
Tylenol Extra Strength - ₪24


#Step 4 — Multi-Tool Agent

This step extends the previous RAG pipeline into an iterative multi-tool agent. Instead of selecting one tool and immediately answering, the LLM can repeatedly select a tool, receive its result as an observation, and use that information to decide what to do next. This allows the agent to chain multiple tools, such as searching for the cheapest product and then using the calculator to determine how many units can be purchased with a given budget. The loop continues until the agent calls final_answer or reaches the maximum number of allowed steps.

In [ ]:
# ============================================================
# Step 4: Multi-Tool Agent
# ============================================================
# Iteratively chooses tools, observes their results, and
# decides what to do next until it produces a final answer.
#
# User → LLM → Tool → Observation → LLM → Tool → ... → Answer
#
# max_steps limits how many iterations the agent can perform.
# The LLM stops early by calling final_answer.
# ============================================================

import json

COLS = [c for c in df.columns if not str(c).startswith("_")]
CATEGORIES = sorted(df["Category"].dropna().unique())

def llm(messages, max_new_tokens=300):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


# ---------------- tools: each one returns a STRING for the model to read ----------------

def show(sub, limit=12):
    """Format matching rows, cheapest first."""
    if sub.empty:
        return "No matching products."
    sub = sub.sort_values("_numeric_price").head(limit)
    return "\n".join(" | ".join(f"{c}: {r[c]}" for c in COLS) for _, r in sub.iterrows())

def search_products(text):
    """Substring search over every column - good for brands and product names."""
    blob = df[COLS].astype(str).agg(" ".join, axis=1).str.lower()
    return show(df[blob.str.contains(str(text).lower(), regex=False)])

def table_query(code):
    """Adapter: Step 3's query_table names its parameter `python_code`, the model writes `code`."""
    return query_table(code)

def by_category(category, max_price=None):
    """All products in a category, optionally under a price."""
    # The model sometimes sends max_price=true when the question had no budget.
    try:
        max_price = None if isinstance(max_price, bool) else float(max_price)
    except (TypeError, ValueError):
        max_price = None
    hit = df["Category"].astype(str).str.lower().str.contains(str(category).lower(), regex=False)
    if not hit.any():
        return f"No such category. Valid categories: {CATEGORIES}"
    sub = df[hit]
    if max_price is not None:
        sub = sub[sub["_numeric_price"] <= float(max_price)]
        if sub.empty:
            return (f"'{category}' has nothing at or under {max_price} ILS. "
                    "Say so - do not offer a product from another category.")
    return show(sub)

def calculator(expression):
    """Arithmetic only: the character whitelist blocks names, so no code can run here."""
    if not re.fullmatch(r"[\d\s.+\-*/()%]+", str(expression)):
        return "Only arithmetic is allowed."
    return str(eval(str(expression)))

TOOLS = {
    "search_products": search_products,   # brands, product names, exact words
    "by_category":     by_category,       # category + optional budget
    "semantic_search":  semantic_search,  # reused from Step 3 (FAISS + BM25)
    "query_table":      table_query,      # reused from Step 3 (pandas expression)
    "calculator":       calculator,       # math on numbers it already retrieved
}

# If the model writes {"args": "Cetaphil"} instead of {"args": {"text": "Cetaphil"}},
# this says which parameter that bare value belongs to.
FIRST_ARG = {"search_products": "text", "by_category": "category", "semantic_search": "query",
             "query_table": "code", "calculator": "expression", "final_answer": "answer"}

AGENT_SYSTEM = f"""You are a drugstore database agent. Answer ONLY from the database, using tools.

Table `df` has {len(df)} products. Columns: {COLS}
Prices are in ILS; df['_numeric_price'] holds them as plain numbers.
Categories: {CATEGORIES}

TOOLS (call exactly one per turn):
- search_products(text): substring search. Use for brands/names, e.g. "Cetaphil", "Nurofen".
- by_category(category, max_price): products in a category, cheapest first. Best for "something for allergies under 30".
- semantic_search(query): meaning-based search for needs not spelled out in the sheet.
- query_table(code): a pandas expression on df, for counts and statistics.
  e.g. "len(df)", "df['_numeric_price'].mean()", "df.loc[df['_numeric_price'].idxmax()]"
- calculator(expression): arithmetic on numbers you already found, e.g. "100 / 14".
- final_answer(answer): give the user the finished answer.

Symptom words are NOT in the sheet: pain relievers are written as "Ibuprofen pain/fever relief"
in category "Pain Relief". For symptoms use by_category or semantic_search, never a literal search.

Reply with ONE JSON object and nothing else:
{{"tool": "<name>", "args": {{...}}}}

Never invent a product or a price - every fact must come from a tool result.
If nothing matches, say so plainly instead of suggesting an unrelated product.
Do not do arithmetic in your head - use calculator and report its number exactly.
Match the product's "Primary Use" to what the user needs, not only the price."""


def run_agent(question, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": AGENT_SYSTEM},
                {"role": "user", "content": question}]

    tool_names = list(TOOLS) + ["final_answer"]
    have_facts = False
    last_call = None

    for step in range(max_steps):
        if step == max_steps - 1:               # don't let it run out of steps silently
            messages.append({"role": "user", "content":
                             "Last turn: reply with final_answer now, using the facts you have."})
        raw = llm(messages)
        try:                                    # greedy {...} = the whole JSON object
            call = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group(0))
        except Exception:
            call = {}
        name = call.get("tool")
        args = call.get("args") or {}
        if not isinstance(args, dict):          # args came as a bare value, not an object
            args = {FIRST_ARG.get(name, "text"): args}

        if verbose:
            print(f"  🔧 step {step + 1}: {name}({args})")

        if name == "final_answer":
            return str(args.get("answer") or raw)

        if name not in TOOLS:
            # No usable JSON. If it already has facts it is just answering in
            # prose instead of calling final_answer - accept that as the answer.
            if have_facts and len(raw) > 40 and not raw.lstrip().startswith("{"):
                return raw
            obs = f"Not a valid call. Reply with ONE JSON object using a tool from: {tool_names}"
        else:
            try:
                obs = str(TOOLS[name](**args))[:2000]
                have_facts = True
            except Exception as e:              # wrong arguments
                obs = f"Error: {e}. Reply with ONE JSON object using a tool from: {tool_names}"

        if call == last_call:                   # it is stuck repeating itself
            obs += "\n(You already made this exact call. Call final_answer now.)"
        last_call = call

        if verbose:
            print(f"     👀 {obs[:150]}...")

        messages += [{"role": "assistant", "content": json.dumps(call)},
                     {"role": "user", "content": f"OBSERVATION:\n{obs}\n\nNext JSON call?"}]

    # Out of steps: answer from the observations gathered instead of giving up.
    if verbose:
        print("     ⏱️  out of steps - summarizing what it found")
    return llm(messages + [{"role": "user", "content":
                            "Answer the user's question now, in plain text, using only the "
                            "observations above. If they contain nothing relevant, say so."}])


# ============================================================
# Test the agent
# ============================================================

for q in [
    "I have a headache. What is the cheapest option, and how many can I buy with 100 ILS?",
    "Is there anything cheaper than Nurofen for the same purpose?",
    "What products under 20 ILS help with allergies?",   # every allergy item is 25+, so: none
    "How many products do you have, and what is the most expensive one?",
    "What Cetaphil products do you carry?",
]:
    print(f"\n❓ {q}")
    print(f"💡 {run_agent(q)}")
    print("-" * 70)



❓ I have a headache. What is the cheapest option, and how many can I buy with 100 ILS?
  🔧 step 1: by_category({'category': 'Pain Relief', 'max_price': True})
     👀 Item: Aspirin 100 mg | Category: Pain Relief | Brand: Aspirin | Size / Pack: 30 tablets | Primary Use: Low-dose aspirin product | Typical Price (ILS):...
  🔧 step 2: calculator({'expression': '100 / 12'})
     👀 8.333333333333334...
  🔧 step 3: final_answer({'answer': 'The cheapest option for headache relief is Aspirin 100 mg at 12 ILS per pack. With 100 ILS, you can buy approximately 8 units.'})
💡 The cheapest option for headache relief is Aspirin 100 mg at 12 ILS per pack. With 100 ILS, you can buy approximately 8 units.
----------------------------------------------------------------------

❓ Is there anything cheaper than Nurofen for the same purpose?
  🔧 step 1: search_products({'text': 'Nurofen'})
     👀 Item: Nurofen 200 mg Tablets | Category: Pain Relief | Brand: Nurofen | Size / Pack: 24 tablets | Primary Use: Ib

#Step 5 — GUI Drugstore Assistant

Creates an interactive Colab GUI using ipywidgets, allowing users to type questions or select example queries and receive answers from the multi-tool agent. It also provides an option to display the agent’s tool execution steps while processing each question.

In [ ]:
# ============================================================
# Step 5: GUI - type a question, get an answer
# ============================================================
# Runs inside the Colab output cell using ipywidgets (no server needed).
# Requires Steps 1-4 to have been run: it calls run_agent().

import io, html, contextlib
import ipywidgets as widgets
from IPython.display import display

EXAMPLES = [
    "I have a headache, what is the cheapest option?",
    "What products under 30 ILS help with allergies?",
    "What Cetaphil products do you carry?",
    "What is the most expensive item you sell?",
]

question_box = widgets.Text(
    placeholder="Ask about the drugstore products...",
    layout=widgets.Layout(width="100%", margin="0 8px 0 0"),
)
ask_button   = widgets.Button(description="Ask", button_style="primary",
                              layout=widgets.Layout(width="90px"))
show_steps   = widgets.Checkbox(value=True, description="Show the agent's tool steps", indent=False)
answer_panel = widgets.HTML()

_CARD = ("border:1px solid #d0d7de; border-radius:8px; padding:12px 14px;"
         "background:#ffffff; color:#111111; font-family:system-ui,sans-serif; line-height:1.5;")

def _render(question, answer, trace):
    out = (f'<div style="{_CARD}">'
           f'<div style="color:#57606a; font-size:13px;">❓ {html.escape(question)}</div>'
           f'<div style="white-space:pre-wrap; margin-top:8px;">{html.escape(answer)}</div>')
    if trace.strip() and show_steps.value:
        out += ('<details style="margin-top:10px;">'
                '<summary style="cursor:pointer; color:#57606a; font-size:13px;">tool steps</summary>'
                f'<pre style="white-space:pre-wrap; font-size:12px; color:#24292f; background:#f6f8fa;'
                f' padding:8px; border-radius:6px; margin-top:6px;">{html.escape(trace.strip())}</pre>'
                '</details>')
    return out + "</div>"

def _status(msg):
    return f'<div style="{_CARD} color:#57606a;">{html.escape(msg)}</div>'

def on_ask(_=None):
    question = question_box.value.strip()
    if not question:
        answer_panel.value = _status("Type a question first.")
        return

    ask_button.disabled = question_box.disabled = True
    answer_panel.value = _status("Thinking... (the agent may call several tools)")

    buffer = io.StringIO()
    try:
        # run_agent prints its tool trace; capture it instead of losing it outside the widget
        with contextlib.redirect_stdout(buffer):
            answer = run_agent(question, verbose=True)
    except Exception as e:
        answer = f"Something went wrong: {type(e).__name__}: {e}"
    finally:
        ask_button.disabled = question_box.disabled = False

    answer_panel.value = _render(question, answer, buffer.getvalue())

def _use_example(text):
    def handler(_):
        question_box.value = text
        on_ask()
    return handler

ask_button.on_click(on_ask)

# Let Enter submit. on_submit is the only API that fires on Enter alone, but it is
# deprecated in ipywidgets 8 - silence that notice, and fall back to observing the
# value (fires on Enter or on losing focus) if a future version drops it.
import warnings
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        question_box.on_submit(on_ask)
except Exception:
    question_box.continuous_update = False
    question_box.observe(lambda change: on_ask(), names="value")

example_buttons = []
for text in EXAMPLES:
    b = widgets.Button(description=text[:38] + ("..." if len(text) > 38 else ""),
                       tooltip=text, layout=widgets.Layout(width="auto", margin="0 4px 4px 0"))
    b.on_click(_use_example(text))
    example_buttons.append(b)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>🏥 Drugstore Assistant</h3>"),
    widgets.HBox([question_box, ask_button]),
    show_steps,
    widgets.HTML("<div style='color:#57606a; font-size:12px; margin-top:6px;'>Try one:</div>"),
    widgets.Box(example_buttons, layout=widgets.Layout(flex_flow="row wrap", display="flex")),
    answer_panel,
], layout=widgets.Layout(width="100%", max_width="760px")))
